# Generic Parameter Robustness Heatmaps

This notebook scans GeoPAS BBOB result directories under `results/bbob_by_deepela/results/bbob/<protocol>/`, filters them by a user-specified `USER_FILTERS` dictionary, parses the aggregate `res_*.csv` summaries, and plots heatmaps over any two user-chosen run parameters.

The current run signature is `scale..._head2scale..._sigmoidlogs..._dual..._head2lw..._head2sw..._priorscale..._lamprior..._tailscale...`, so any parameter you are not plotting should usually stay fixed in `USER_FILTERS`. If one of the plotted parameters appears in `USER_FILTERS`, leave it as `None`.

Suggested workflow:
1. Set `PLOT_Y_PARAM`, `PLOT_X_PARAM`, and `USER_FILTERS` in Cell 2.
2. Run Cell 3 to inspect the matching fixed-parameter slices and tighten the remaining filters until the desired slice is isolated.
3. Run Cell 4 to build `df_grid` and confirm how many unique values were loaded for the two plotted parameters.
4. Run Cell 5 once per kernel session to load the plotting helpers.
5. Run Cells 6 and 7 to generate the summary and bounded-performance heatmaps.

In [4]:
from __future__ import annotations

import pandas as pd

try:
    from analyses.analysis_utils import (
        RUN_PARAMETER_COLUMNS,
        apply_paper_theme,
        collect_parameter_grid_results,
        list_available_parameter_slices,
        plot_parameter_heatmap,
        resolve_geopas_paths,
    )
except ModuleNotFoundError:
    from analysis_utils import (
        RUN_PARAMETER_COLUMNS,
        apply_paper_theme,
        collect_parameter_grid_results,
        list_available_parameter_slices,
        plot_parameter_heatmap,
        resolve_geopas_paths,
    )

PATHS = resolve_geopas_paths()
RESULTS_ROOT = PATHS.bbob_results_root
SAVE_DIR = PATHS.analysis_output_root / "parameter_heatmaps"

PLOT_Y_PARAM = "sigmoid_log_s"
PLOT_X_PARAM = "lam_prior"

# Leave the plotted axes as None so the notebook scans the full slice.
USER_FILTERS = {
    "protocol": "lio",
    "target_scale": "log_power",
    "head_2_target_scale": "norm",
    "prior_scale": "log_power",
    "sigmoid_log_s": None,
    "tail_scale": 1.0,
    "lam_prior": None,
    "dual_head": 0,
    "head_2_loss_weight": 0.0,
    "head_2_score_weight": 0.0,
    "res": 8,
    "k_views": 32,
    "seed": None,
}

apply_paper_theme()

In [5]:
invalid_plot_params = [column for column in (PLOT_Y_PARAM, PLOT_X_PARAM) if column not in RUN_PARAMETER_COLUMNS]
if invalid_plot_params:
    print(f"Unknown plot parameters: {invalid_plot_params}")
    print(f"Choose from: {', '.join(RUN_PARAMETER_COLUMNS)}")
    available_slices = pd.DataFrame()
elif PLOT_X_PARAM == PLOT_Y_PARAM:
    print("Choose different values for PLOT_X_PARAM and PLOT_Y_PARAM.")
    available_slices = pd.DataFrame()
else:
    available_slices = list_available_parameter_slices(
        USER_FILTERS,
        varying_cols=(PLOT_Y_PARAM, PLOT_X_PARAM),
        results_root=RESULTS_ROOT,
    )
    if available_slices.empty:
        print("No matching fixed-parameter slices were found for the current USER_FILTERS.")
    else:
        total_pairs = int(available_slices["n_plot_parameter_pairs"].sum())
        print(
            f"Found {len(available_slices)} fixed-parameter slice(s), covering {total_pairs} ({PLOT_Y_PARAM}, {PLOT_X_PARAM}) pair(s)."
        )

available_slices

Found 1 fixed-parameter slice(s), covering 154 (sigmoid_log_s, lam_prior) pair(s).


,target_scale,head_2_target_scale,prior_scale,tail_scale,dual_head,head_2_loss_weight,head_2_score_weight,res,k_views,n_plot_parameter_pairs
0,log_power,norm,log_power,1.0,0,0.0,0.0,8,32,154


In [6]:
df_grid = None
if invalid_plot_params:
    print(f"Unknown plot parameters: {invalid_plot_params}")
elif PLOT_X_PARAM == PLOT_Y_PARAM:
    print("Choose different values for PLOT_X_PARAM and PLOT_Y_PARAM.")
elif USER_FILTERS.get(PLOT_X_PARAM) is not None or USER_FILTERS.get(PLOT_Y_PARAM) is not None:
    print(f"Set USER_FILTERS[{PLOT_X_PARAM!r}] and USER_FILTERS[{PLOT_Y_PARAM!r}] to None, then rerun this cell.")
else:
    df_grid = collect_parameter_grid_results(
        USER_FILTERS,
        x_col=PLOT_X_PARAM,
        y_col=PLOT_Y_PARAM,
        results_root=RESULTS_ROOT,
    )
    if df_grid.empty:
        print("No matching runs were found. Adjust USER_FILTERS after checking available_slices, then rerun this cell.")
        df_grid = None
    else:
        print(
            "Loaded "
            f"{len(df_grid)} runs across "
            f"{df_grid[PLOT_Y_PARAM].nunique()} {PLOT_Y_PARAM} values x "
            f"{df_grid[PLOT_X_PARAM].nunique()} {PLOT_X_PARAM} values."
        )
        display_columns = [
            column
            for column in [PLOT_Y_PARAM, PLOT_X_PARAM, "mean", "median", "p90", "accuracy", "f1"]
            if column in df_grid.columns
        ]
        df_grid[display_columns]

Loaded 154 runs across 14 sigmoid_log_s values x 11 lam_prior values.


In [5]:
SUMMARY_PLOTS = [
    ("mean", "Mean"),
    ("median", "Median"),
    ("p90", "P90"),
]

if df_grid is None:
    print("Adjust PLOT_X_PARAM, PLOT_Y_PARAM, and USER_FILTERS, rerun Cell 4, then rerun this cell.")
else:
    for column, title in SUMMARY_PLOTS:
        plot_title = f"{title} (log-scale colour)" if column in {"mean", "p90"} else title
        plot_parameter_heatmap(
            df_grid,
            column,
            plot_title,
            x_col=PLOT_X_PARAM,
            y_col=PLOT_Y_PARAM,
            output_path=SAVE_DIR,
            log_scale=column in {"mean", "p90"},
        )

Adjust PLOT_X_PARAM, PLOT_Y_PARAM, and USER_FILTERS, rerun Cell 4, then rerun this cell.


In [6]:
BOUNDED_PLOTS = [
    ("accuracy", "Accuracy"),
    ("f1", "F1"),
]

if df_grid is None:
    print("Adjust PLOT_X_PARAM, PLOT_Y_PARAM, and USER_FILTERS, rerun Cell 4, then rerun this cell.")
else:
    missing_cols = [column for column, _ in BOUNDED_PLOTS if column not in df_grid.columns]
    if missing_cols:
        print(f"Skipping missing columns: {missing_cols}")

    for column, title in BOUNDED_PLOTS:
        if column not in df_grid.columns:
            continue
        # if column == "accuracy":
        #     plot_parameter_heatmap(
        #         df_grid,
        #         column,
        #         title,
        #         x_col=PLOT_X_PARAM,
        #         y_col=PLOT_Y_PARAM,
        #     )
        # else:
        #     plot_parameter_heatmap(
        #         df_grid,
        #         column,
        #         title,
        #         x_col=PLOT_X_PARAM,
        #         y_col=PLOT_Y_PARAM,
        #         output_path=SAVE_DIR,
        #     )
        
        plot_parameter_heatmap(
            df_grid,
            column,
            title,
            x_col=PLOT_X_PARAM,
            y_col=PLOT_Y_PARAM,
            output_path=SAVE_DIR,
        )

Adjust PLOT_X_PARAM, PLOT_Y_PARAM, and USER_FILTERS, rerun Cell 4, then rerun this cell.
